# Chapter 6 — Data Collection
### Companion Notebook

*Data Science for Business and Economics: A Python Approach* — Dr. Jorge Zazueta Gutiérrez

This notebook mirrors every code example in Chapter 6. The API cells below call the real,
live Frankfurter currency API (https://api.frankfurter.dev) — no mocking, no fictional
provider. Values you get back will differ from whatever is printed in the chapter, since
exchange rates move every business day; that is expected, and the chapter says so explicitly.
This notebook needs an internet connection to run the API cells.

The scraping cells read a saved snapshot of a SIAP-style market page (`siap_spot_page.html`,
included alongside this notebook) rather than a live URL, so the scraping mechanics stay
runnable even though no single specific webpage is guaranteed to still exist, unchanged,
whenever you're reading this.

In [ ]:
book_metadata = {
    "title": "Data Science for Business and Economics: A Python Approach",
    "chapter": 6,
    "chapter_title": "Data Collection",
    "author": "Jorge Zazueta Gutierrez",
    "protagonist": "Renata Solis",
    "part": 2,
    "part_title": "Methods and Models",
    "requires": ["pandas", "numpy", "requests", "beautifulsoup4",
                 "matplotlib", "seaborn"],
    "notebook_version": "0.2",
}
book_metadata

## 6.2 APIs: Structured Access

### 6.2.3 A First Real Request

This cell makes a genuine network request to a real server. No mocking.

In [ ]:
import requests
import pandas as pd

response = requests.get(
    "https://api.frankfurter.dev/v1/latest",
    params={"base": "USD", "symbols": "MXN"}
)
print(response.status_code)
print(response.url)

In [ ]:
data = response.json()
data

In [ ]:
usd_mxn_rate = data["rates"]["MXN"]
usd_mxn_rate

### 6.2.4 Status Codes and Error Handling

In [ ]:
response = requests.get(
    "https://api.frankfurter.dev/v1/latest",
    params={"base": "USD", "symbols": "MXN"}
)
response.raise_for_status()
data = response.json()
print("ok")

### 6.2.5 Application: Thirty Days of the Peso Exchange Rate

In [ ]:
response = requests.get(
    "https://api.frankfurter.dev/v1/2026-06-01..2026-06-30",
    params={"base": "USD", "symbols": "MXN"}
)
response.raise_for_status()
history = response.json()
print(history["start_date"], "to", history["end_date"])
print(list(history["rates"].items())[:3])

In [ ]:
fx_history = pd.DataFrame(history["rates"]).T
fx_history.index.name = "date"
fx_history = fx_history.reset_index()
print(fx_history.shape)
print(fx_history.dtypes)
fx_history.head()

**Bonus: plot the series.** Not in the printed chapter, but a natural next step once
`fx_history` exists.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fx_history["date"] = pd.to_datetime(fx_history["date"])
sns.set_style("white")
fig, ax = plt.subplots(figsize=(6.5, 4))
sns.lineplot(data=fx_history, x="date", y="MXN", color="#4682B4",
             marker="o", markersize=4, ax=ax)
ax.set_ylabel("USD/MXN rate")
ax.set_xlabel("Date")
ax.tick_params(axis="x", rotation=30)
sns.despine()
plt.tight_layout()
plt.show()

## 6.3 Web Scraping: When There Is No API

### Reading HTML as Structure

In [ ]:
from bs4 import BeautifulSoup

html = """
<table id="prices">
  <tr><th>Product</th><th>Price</th></tr>
  <tr><td>Corn</td><td>4.12</td></tr>
  <tr><td>Wheat</td><td>5.87</td></tr>
</table>
"""

soup = BeautifulSoup(html, "html.parser")
rows = soup.find("table", id="prices").find_all("tr")[1:]

records = [
    {"product": r.find_all("td")[0].text,
     "price": float(r.find_all("td")[1].text)}
    for r in rows
]
pd.DataFrame(records)

### Application: Scraping Wholesale Spot Prices

Reads a saved snapshot (`siap_spot_page.html`) rather than a live URL, for the reason the
chapter explains: no single webpage is a safe permanent dependency for a textbook example.
Swap the `open(...)` call for `requests.get(url, ...)` to run this against a real live page.

In [ ]:
with open("siap_spot_page.html") as f:
    html_page = f.read()

# To pull the live page instead, replace the two lines above with:
#     response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
#     response.raise_for_status()
#     html_page = response.text

soup = BeautifulSoup(html_page, "html.parser")
table = soup.find("table", class_="precios-diarios")

rows = table.find_all("tr")[1:]
records = []
for r in rows:
    cells = r.find_all("td")
    records.append({
        "state": cells[0].text.strip(),
        "market": cells[1].text.strip(),
        "price_mxn_kg": cells[2].text.strip(),
    })

spot_raw = pd.DataFrame(records)
spot_raw

## 6.4 From Raw Pull to Usable Table

In [ ]:
spot_raw["price_mxn_kg"] = pd.to_numeric(
    spot_raw["price_mxn_kg"], errors="coerce"
)
print(spot_raw.dtypes)
print(spot_raw["price_mxn_kg"].isnull().sum())

In [ ]:
spot_daily = (
    spot_raw
    .assign(date="2026-06-30")
    .groupby("date")["price_mxn_kg"]
    .mean()
    .reset_index(name="spot_price_mxn_kg")
)

# uses the real fx_history pulled live in Section 6.2
day_rate = fx_history.loc[fx_history["date"] == "2026-06-30", "MXN"].item()
spot_daily["spot_price_usd_kg"] = (
    spot_daily["spot_price_mxn_kg"] / day_rate
).round(2)
spot_daily

In [ ]:
tracker = fx_history.merge(
    spot_daily[["date", "spot_price_usd_kg"]],
    on="date", how="left"
)
tracker.tail(3)

## Exercises

Work through each exercise in its own cell below.

**Exercise 1 (APIs).** Write a version of the range-request cell (listing 6.5) that catches a failed request, prints a clear message, and falls back to requesting just the single most recent date.

In [ ]:
# Exercise 1


**Exercise 2 (HTML structure).** Using a browser's Inspect Element tool on a public data table of your choice, identify the `id` or `class` that distinguishes the table you want. Write your answer here.

_Your answer:_

**Exercise 3 (Web scraping).** Add a check immediately after the scrape in Section 6.3 that detects whether the page's table is empty or missing.

In [ ]:
# Exercise 3


**Exercise 4 (Integration).** Re-run the merge above with `how="inner"` instead of `how="left"`. Compare the resulting shape to the original.

In [ ]:
# Exercise 4


**Exercise 5 (A different pair).** Adapt the Section 6.2 range-query code to pull the USD/EUR rate instead of USD/MXN, over the same window. Plot both series together.

In [ ]:
# Exercise 5


**Exercise 6 (Synthesis).** Sketch, in a short paragraph, what a fully automated version of Renata's tracker would need beyond what this notebook builds.

_Your answer:_